# Lab 5: RAG Evaluation — Groundedness, Relevance & Coherence

**Duration:** 30 minutes  
**Environment:** Azure ML Studio Notebook  

---

## Learning Objectives

- Build a test dataset of banking Q&A pairs
- Evaluate RAG responses for groundedness, relevance, coherence
- Generate an evaluation report with quality assessment

## Prerequisites

Labs 1-3 completed

---

© 2026 Great Learning. All rights reserved.

## Step 1: Load Configuration

In [ ]:
import os, json, urllib.request, ssl, subprocess

RESOURCE_GROUP = 'rg-promptflow-rag-lab'

# Auto-detect OpenAI resource that has BOTH gpt-4o and text-embedding-3-small deployments
# This handles the case where Lab 1 was run multiple times creating multiple OpenAI resources
print('Detecting Azure resources...')
r = subprocess.run(['az', 'cognitiveservices', 'account', 'list', '-g', RESOURCE_GROUP,
    '--query', "[?kind=='OpenAI'].name", '-o', 'tsv'], capture_output=True, text=True)
openai_candidates = [n.strip() for n in r.stdout.strip().split('\n') if n.strip()]

OPENAI_NAME = ''
for candidate in openai_candidates:
    # Check if this resource has both required deployments
    r = subprocess.run(['az', 'cognitiveservices', 'account', 'deployment', 'list',
        '-n', candidate, '-g', RESOURCE_GROUP, '--query', '[].name', '-o', 'tsv'],
        capture_output=True, text=True)
    deployments = r.stdout.strip().split('\n')
    if 'gpt-4o' in deployments and 'text-embedding-3-small' in deployments:
        OPENAI_NAME = candidate
        print(f'  Found OpenAI with deployments: {candidate}')
        break

if not OPENAI_NAME:
    print('ERROR: No OpenAI resource found with both gpt-4o and text-embedding-3-small deployments.')
    print(f'  Candidates checked: {openai_candidates}')
    print('  Fix: Go back to Lab 1 and run Steps 4 + 5 to create deployments.')
    raise SystemExit(1)

# Auto-detect Search service
r = subprocess.run(['az', 'search', 'service', 'list', '-g', RESOURCE_GROUP,
    '--query', '[].name', '-o', 'tsv'], capture_output=True, text=True)
search_candidates = [n.strip() for n in r.stdout.strip().split('\n') if n.strip()]
SEARCH_NAME = search_candidates[0] if search_candidates else ''

# Auto-detect Storage account
r = subprocess.run(['az', 'storage', 'account', 'list', '-g', RESOURCE_GROUP,
    '--query', '[0].name', '-o', 'tsv'], capture_output=True, text=True)
STORAGE_NAME = r.stdout.strip()

if not SEARCH_NAME:
    print('ERROR: No Search service found. Run Lab 1 first.')
    raise SystemExit(1)

# Get endpoints and keys
r = subprocess.run(['az', 'cognitiveservices', 'account', 'show', '-n', OPENAI_NAME, '-g', RESOURCE_GROUP,
    '--query', 'properties.endpoint', '-o', 'tsv'], capture_output=True, text=True)
OPENAI_ENDPOINT = r.stdout.strip()
if not OPENAI_ENDPOINT.endswith('/'): OPENAI_ENDPOINT += '/'

r = subprocess.run(['az', 'cognitiveservices', 'account', 'keys', 'list', '-n', OPENAI_NAME, '-g', RESOURCE_GROUP,
    '--query', 'key1', '-o', 'tsv'], capture_output=True, text=True)
OPENAI_KEY = r.stdout.strip()

SEARCH_ENDPOINT = f'https://{SEARCH_NAME}.search.windows.net'
r = subprocess.run(['az', 'search', 'admin-key', 'show', '--service-name', SEARCH_NAME, '-g', RESOURCE_GROUP,
    '--query', 'primaryKey', '-o', 'tsv'], capture_output=True, text=True)
SEARCH_KEY = r.stdout.strip()

ctx = ssl.create_default_context()

print(f'\n✅ Resources detected:')
print(f'  OpenAI:  {OPENAI_NAME}')
print(f'  Search:  {SEARCH_NAME}')
print(f'  Storage: {STORAGE_NAME}')
print(f'  Endpoint: {OPENAI_ENDPOINT}')

## Step 2: Create Banking Test Dataset

8 Q&A pairs covering policies, products, and security.

In [ ]:
test_dataset = [
    {'question':'What is the daily wire transfer limit for personal accounts?',
     'expected_answer':'$50,000 domestic, $25,000 international.','category':'policy'},
    {'question':'How long do I have to dispute a credit card transaction?',
     'expected_answer':'Within 60 days of the statement date.','category':'policy'},
    {'question':'What is the APY on the High-Yield Savings Account?',
     'expected_answer':'4.25% APY with $10,000 minimum balance.','category':'product'},
    {'question':'What should I do if I suspect fraud on my account?',
     'expected_answer':'Call 1-800-SECURE-1, lock card, file report.','category':'security'},
    {'question':'What is the interest rate for a 12-month CD?',
     'expected_answer':'5.00% APY with $1,000 minimum.','category':'product'},
    {'question':'Is dual authorization required for business wire transfers?',
     'expected_answer':'Yes, for amounts over $50,000.','category':'policy'},
    {'question':'What is the zero liability protection policy?',
     'expected_answer':'Zero liability for unauthorized transactions reported within 2 business days.','category':'security'},
    {'question':'What are the personal loan APR ranges?',
     'expected_answer':'6.99% to 15.99% based on credit score, 12-60 months.','category':'product'},
]

print(f'✅ Test dataset: {len(test_dataset)} Q&A pairs')
for i, t in enumerate(test_dataset, 1):
    print(f'   {i}. [{t["category"]}] {t["question"][:60]}...')

## Step 3: Define Helper Functions

In [ ]:
def call_openai(messages, max_tokens=150, temperature=0.1):
    url = f"{OPENAI_ENDPOINT}openai/deployments/gpt-4o/chat/completions?api-version=2024-06-01"
    data = json.dumps({'messages':messages,'max_tokens':max_tokens,'temperature':temperature}).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type':'application/json','api-key':OPENAI_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read())['choices'][0]['message']['content']

def get_embedding(text):
    url = f"{OPENAI_ENDPOINT}openai/deployments/text-embedding-3-small/embeddings?api-version=2024-06-01"
    data = json.dumps({'input':text}).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type':'application/json','api-key':OPENAI_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read())['data'][0]['embedding']

def hybrid_search(query, top_k=3):
    vector = get_embedding(query)
    url = f"{SEARCH_ENDPOINT}/indexes/banking-policies/docs/search?api-version=2024-07-01"
    body = {'search':query,'vectorQueries':[{'vector':vector,'k':top_k,'fields':'content_vector','kind':'vector'}],
            'select':'title,content,category,source_file','top':top_k}
    data = json.dumps(body).encode()
    req = urllib.request.Request(url, data=data, method='POST',
        headers={'Content-Type':'application/json','api-key':SEARCH_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read()).get('value', [])

def rag_answer(question):
    results = hybrid_search(question)
    context = '\n\n'.join([f'[{r["title"]}]: {r["content"]}' for r in results])
    sources = [r['source_file'] for r in results]
    response = call_openai([{'role':'system','content':'Banking assistant. Answer ONLY from context. Cite sources.'},
        {'role':'user','content':f'Context:\n{context}\n\nQuestion: {question}'}])
    return response, context, sources

print('✅ Helper functions defined.')

## Step 4: Define Evaluation Metrics

Four metrics scored 1-5 by GPT-4o:
- **Groundedness** — Is the answer supported by the context?
- **Relevance** — Does the answer address the question?
- **Coherence** — Is the answer well-structured?
- **Correctness** — Does it match the expected answer?

In [ ]:
def evaluate_metric(metric, question, answer, context='', expected=''):
    prompts = {
        'groundedness': f'Rate groundedness 1-5. Is every claim supported by context?\nContext: {context[:2000]}\nQuestion: {question}\nAnswer: {answer}\nScore (1-5):',
        'relevance': f'Rate relevance 1-5. Does the answer address the question?\nQuestion: {question}\nAnswer: {answer}\nScore (1-5):',
        'coherence': f'Rate coherence 1-5. Is the answer well-structured and clear?\nAnswer: {answer}\nScore (1-5):',
        'correctness': f'Rate correctness 1-5. Does generated match expected?\nExpected: {expected}\nGenerated: {answer}\nScore (1-5):',
    }
    score = call_openai([{'role':'user','content':prompts[metric]}], max_tokens=5, temperature=0)
    try: return int(score.strip()[0])
    except: return 3

print('✅ Evaluation metrics defined.')

## Step 5: Run Evaluation on Test Dataset

In [ ]:
print('=' * 70)
print('  RAG EVALUATION REPORT — SecureBank Banking Assistant')
print('=' * 70)

results = []
for i, test in enumerate(test_dataset, 1):
    print(f'\n--- Test {i}/{len(test_dataset)}: {test["question"][:50]}... ---')
    answer, context, sources = rag_answer(test['question'])
    print(f'  Answer: {answer[:100]}...')
    g = evaluate_metric('groundedness', test['question'], answer, context)
    r = evaluate_metric('relevance', test['question'], answer)
    c = evaluate_metric('coherence', test['question'], answer)
    x = evaluate_metric('correctness', test['question'], answer, expected=test['expected_answer'])
    print(f'  Scores: Ground={g}/5  Relev={r}/5  Coher={c}/5  Correct={x}/5')
    results.append({'question':test['question'],'category':test['category'],
        'groundedness':g,'relevance':r,'coherence':c,'correctness':x,
        'avg':round((g+r+c+x)/4, 1)})
    time.sleep(0.5)

# Summary table
print('\n' + '=' * 70)
print('  EVALUATION SUMMARY')
print('=' * 70)
print(f'{"#":<4} {"Category":<10} {"Ground":<8} {"Relev":<8} {"Coher":<8} {"Correct":<9} {"Avg":<6}')
print('-' * 53)
tg, tr, tc, tx = 0, 0, 0, 0
for i, res in enumerate(results, 1):
    print(f'{i:<4} {res["category"]:<10} {res["groundedness"]:<8} {res["relevance"]:<8} {res["coherence"]:<8} {res["correctness"]:<9} {res["avg"]:<6}')
    tg += res['groundedness']; tr += res['relevance']; tc += res['coherence']; tx += res['correctness']
n = len(results)
avg_all = (tg+tr+tc+tx)/(4*n)
print('-' * 53)
print(f'{"AVG":<15} {tg/n:<8.1f} {tr/n:<8.1f} {tc/n:<8.1f} {tx/n:<9.1f} {avg_all:<6.1f}')

if avg_all >= 4.0: print(f'\n🟢 EXCELLENT — avg: {avg_all:.1f}/5')
elif avg_all >= 3.0: print(f'\n🟡 GOOD — avg: {avg_all:.1f}/5')
else: print(f'\n🔴 NEEDS WORK — avg: {avg_all:.1f}/5')

## ✅ Lab 5 Complete

**What you accomplished:**
- Created 8-question banking test dataset
- Evaluated groundedness, relevance, coherence, correctness
- Generated evaluation report with quality assessment

**Next:** Open `promptflow_lab6_prompt_variants.ipynb`


> 🧹 **Cleanup:** Run the cleanup cell in **Lab 9** when all labs are complete to delete all resources.
---

© 2026 Great Learning. All rights reserved.